# IOAI — 2025 Individual Contest Antique — ⭐모범답안 (Colab 자동 설정판)

이 노트북은 IOAI 로컬 연습 사이트에서 **데이터·학습환경이 자동 준비**되도록 생성되었습니다.
아래 **설정 셀을 먼저 실행**하면 공식 GitHub 저장소에서 이 문제 폴더만 부분 클론으로 받아
(전체 6.6GB 가 아니라 해당 폴더만), 그 폴더로 이동한 뒤 이후 셀이 그대로 학습/예측을 합니다.
완료 후 생성되는 제출 파일을 내려받아 연습 사이트의 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** 로 바꾸면 학습이 빨라집니다.

In [ ]:
# === 데이터 + 환경 자동 설정 (가장 먼저 실행) ===
# 공식 공개 저장소에서 이 문제 폴더만 부분 클론(sparse)으로 받고 그 폴더로 이동한다.
import os
REPO_URL = "https://github.com/IOAI-official/IOAI-2025"
CLONE = "IOAI-2025"
SUBDIR = "Individual-Contest/Antique"
# Colab 은 /content 가 홈. 재실행해도 경로가 안정적이도록 고정 기준에서 시작한다.
BASE = "/content" if os.path.isdir("/content") else os.getcwd()
os.chdir(BASE)
if not os.path.isdir(os.path.join(CLONE, SUBDIR)):
    !git clone --filter=blob:none --no-checkout --depth 1 $REPO_URL $CLONE
    !cd $CLONE && git sparse-checkout set "$SUBDIR"
    !cd $CLONE && git checkout
os.chdir(os.path.join(BASE, CLONE, SUBDIR))
print("작업 폴더:", os.getcwd())
print("내용:", sorted(os.listdir(".")))

<img src="./figs/IOAI-Logo.png" alt="IOAI Logo" width="200" height="auto">

[IOAI 2025 (Beijing, China), Individual Contest](https://ioai-official.org/china-2025)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IOAI-official/IOAI-2025/blob/main/Individual-Contest/Antique/Solution/Antique_Solution.ipynb)

# Antique Painting Authentication: Reference Solution

## Step 1: Train Your Model

In [ ]:
import os
import sys

# 1. Get the current working directory
current_dir = os.getcwd()

# 2. Check if the path contains "Individual-Contest/Antique" and trim it to that point
project_root = current_dir

# 3. Change working directory to the project root
os.chdir(project_root)
print("Working directory set to:", os.getcwd())

# 4. Add module search path (e.g., where metrics.py is located)
sys.path.append(os.path.join(project_root, "Scoring"))

In [ ]:
import pandas as pd
import numpy as np
from sklearn.cluster import SpectralClustering
from collections import Counter
from sklearn.svm import SVC
import os

TRAIN_PATH = "./training_set/" # The address of trainig set

train = pd.read_csv(TRAIN_PATH + "training_set.csv")

X = np.array(train.iloc[:,:5])
y = np.array(train.iloc[:,5])

labeled_mask = y != 0
unlabeled_mask = y == 0
X_labeled = X[labeled_mask]
y_labeled = y[labeled_mask]
X_unlabeled = X[unlabeled_mask]

n_clusters = 2
spectral = SpectralClustering(n_clusters=n_clusters, affinity='rbf', gamma=10, random_state=42)
cluster_labels = spectral.fit_predict(X) 

cluster_to_label = {}
for cluster in range(n_clusters):

    labeled_in_cluster = y_labeled[cluster_labels[labeled_mask] == cluster]

    if len(labeled_in_cluster) > 0:
        most_common_label = Counter(labeled_in_cluster).most_common(1)[0][0]
        cluster_to_label[cluster] = most_common_label

pseudo_labels = np.array([cluster_to_label[cluster] for cluster in cluster_labels])

svm = SVC(kernel='rbf', C=1.0, gamma='scale', random_state=42)
svm.fit(X, pseudo_labels)

## Step 2: Make Predictions on the Validation and Test Set

In [ ]:
VAL_DATA_PATH = "./Solution/validation_set/"
TEST_DATA_PATH = "./Solution/test_set/"

testA = np.array(pd.read_csv(VAL_DATA_PATH + "validation_set.csv"))
testB = np.array(pd.read_csv(TEST_DATA_PATH + "test_set.csv"))

predA = svm.predict(testA)
predB = svm.predict(testB)

## Step 3: Generate `submission.zip` for Submission

In [ ]:
import zipfile
import os

submissionA = pd.DataFrame(predA)
submissionA.to_csv("./Scoring/submissionA.csv", index=False, header=False)

submissionB = pd.DataFrame(predB)
submissionB.to_csv("./Scoring/submissionB.csv", index=False, header=False)

files_to_zip = ['./Scoring/submissionA.csv', './Scoring/submissionB.csv']
zip_filename = './Scoring/submission.zip'

with zipfile.ZipFile(zip_filename, 'w') as zipf:
    for file in files_to_zip:
        zipf.write(file, os.path.basename(file))

print(f'{zip_filename} is created succefully!')

### Evaluate the Model Performance

In [ ]:
%run Scoring/metrics.py

## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.zip']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)